## Sub-Phase 3.1 & Phase 5 Setup (Colab PREREQUISITES)

These cells must be run to hydrate the environment and variables before calling the `Trainer`.

In [ ]:
# 0. Mount Google Drive for Persistent Storage
# This ensures that checkpoints, models, and training logs are saved permanently 
# and won't be lost if the Colab instance times out.
from google.colab import drive
import os

drive.mount('/content/drive')

# Create a dedicated directory in your Drive for this project
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Hiligaynon_NER_Project'
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, 'logs'), exist_ok=True)

print(f"Google Drive mounted. Project data will be saved to: {DRIVE_PROJECT_DIR}")

In [ ]:
# 1. Install Required Libraries
!pip install -q transformers datasets seqeval evaluate spacy sklearn-crfsuite
print("Dependencies installed.")

import os
import numpy as np
import evaluate
from datasets import Dataset, DatasetDict

In [ ]:
# 2. Model & Tokenizer Initialization
# (Ensure model_xlmr.py is uploaded to your Colab directory)
try:
    from model_xlmr import initialize_xlmr_model
except ModuleNotFoundError:
    print("WARNING: 'model_xlmr.py' not found! Make sure to upload it to the Colab runtime.")

label_list = [
    "O", 
    "B-PERSON", "I-PERSON", "E-PERSON", "S-PERSON",
    "B-GPE",    "I-GPE",    "E-GPE",    "S-GPE",
    "B-ORG",    "I-ORG",    "E-ORG",    "S-ORG",
    "B-DATE",   "I-DATE",   "E-DATE",   "S-DATE"
]

print("Initializing Model...")
model, tokenizer, config = initialize_xlmr_model(label_list)

In [ ]:
# 3. Data Extraction (Parsing CoNLL to Hugging Face Datasets)
# (Ensure data/final_train.conll and data/final_test_gold.conll are uploaded)
def read_conll(file_path):
    if not os.path.exists(file_path):
        return {"tokens": [], "ner_tags": []}
    
    with open(file_path, "r", encoding="utf-8") as f:
        sentences, labels, current_words, current_labels = [], [], [], []
        for line in f:
            if line.startswith("-DOCSTART-") or line == "" or line == "\n":
                if current_words:
                    sentences.append(current_words)
                    labels.append(current_labels)
                    current_words, current_labels = [], []
            else:
                splits = line.strip().split("\t")
                if len(splits) >= 2:
                    current_words.append(splits[0])
                    current_labels.append(splits[1])
        if current_words:
            sentences.append(current_words)
            labels.append(current_labels)
    return {"tokens": sentences, "ner_tags": labels}

train_data = read_conll("../data/final_train.conll")
eval_data = read_conll("../data/final_test_gold.conll")

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(eval_data)
})

In [ ]:
# 4. Input Tokenization and Alignment 
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # -100 tells PyTorch to ignore this token during loss calculation
            elif word_idx != previous_word_idx:
                label_ids.append(config.label2id.get(label[word_idx], 0))
            else:
                label_ids.append(-100) # Only label the first subword token
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
train_dataset = tokenized_datasets.get("train")
eval_dataset = tokenized_datasets.get("validation")

In [ ]:
# 5. Entity-Level Evaluation Metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [config.id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [config.id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("Pipeline Setup Complete! Ready to execute training.")

## Sub-Phase 4.1: Environment Setup & Hyperparameter Tuning

**Context:** Resources are constrained to Colab Pro T4/A100. Small datasets are prone to overfitting; meticulous hyperparameter setting is critical.

In [ ]:
import os
import torch
from transformers import TrainingArguments, Trainer

# Set Hyperparameters based on constrained resources (Colab Pro T4/A100)
BATCH_SIZE = 16
LEARNING_RATE = 2e-5  # Optional alternative: 3e-5
WEIGHT_DECAY = 0.01
EPOCHS = 5
WARMUP_RATIO = 0.1

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/Hiligaynon_NER_Project/checkpoints',
    logging_dir='/content/drive/MyDrive/Hiligaynon_NER_Project/logs',
    logging_strategy='steps',
    logging_steps=50,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    # Injecting linear scheduler
    lr_scheduler_type='linear',
    warmup_ratio=WARMUP_RATIO,
    # FP16 enabled if GPU is available (ideal for T4/A100 instances)
    fp16=torch.cuda.is_available(),
)

print("TrainingArguments effectively initialized with MLOps constraints.")

## Sub-Phase 4.2: Execution & Checkpointing

**Context:** Execute the training loop. Ensure models save top-performing epochs based on the localized validation loss. Safeguards are in place via `save_strategy='epoch'` and `load_best_model_at_end=True`.

In [ ]:
from transformers import Trainer, DataCollatorForTokenClassification

# NOTE: Variables like `model`, `tokenizer`, `train_dataset`, and `eval_dataset` 
# are expected to be imported or instantiated from your Data pipelines and Phase 3.1.
# `compute_metrics` will be passed from the evaluation script later (Phase 5.1).

def execute_training(model, tokenizer, train_dataset, eval_dataset, compute_metrics=None):
    """
    Instantiates the Hugging Face Trainer and executes the backpropagation loop.
    Checkpointing is handled automatically via `training_args`.
    """
    # Data collator dynamically pads inputs and labels to the maximum length of a batch
    data_collator = DataCollatorForTokenClassification(tokenizer)
    
    # Initialize the Trainer
    trainer = Trainer(
        model=model,
        args=training_args,                  # Inherited from cell above (Sub-Phase 4.1)
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics      # Entity-level evaluation metrics (seqeval)
    )
    
    print("Initiating training sequence...")
    print("Models will be persistently checkpointed at: /content/drive/MyDrive/Hiligaynon_NER_Project/checkpoints/")
    
    # Execute the core training loop
    # This will generate training runtime logs and intermittent .safetensors 
    # based on validation loss performance.
    trainer.train()
    
    # Because load_best_model_at_end=True, calling save_model will serialize 
    # the top-performing checkpoint structure to our target directory
    final_save_path = "/content/drive/MyDrive/Hiligaynon_NER_Project/checkpoints/best_model"
    trainer.save_model(final_save_path)
    
    print(f"Training finalized. Top-performing model safely preserved at {final_save_path}")
    
    return trainer

# Uncomment the following when running the actual architecture:
# trainer = execute_training(model, tokenizer, train_dataset, eval_dataset, compute_metrics)